In [21]:
import nltk
from nltk.corpus import treebank
from sklearn.model_selection import train_test_split
from collections import defaultdict
import numpy as np

In [22]:
nltk.download('treebank')
nltk.download('universal_tagset')

[nltk_data] Downloading package treebank to /root/nltk_data...
[nltk_data]   Package treebank is already up-to-date!
[nltk_data] Downloading package universal_tagset to /root/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


True

Initial approach:
1. Vanilla HMM (no smoothing and no UNK)
2. HMM (no smoothing, with UNK)
3. HMM (smoothing + UNK)

Then find the accuracy in above scenarios

Approach:


1.   **HMMTagger(Vanilla HMM)**:
  Here unknown words are not explicitly handled. if a word encountered during the testing was not seen during training, it's emission probability for any state will be 0.
2.   **HMMTaggerUNK:**
  an explicit strategy for unknown words implemented.
  **During Training**:
    Words that appear fewer times than specified (unk_thresold) "defaulting to 1" are placed with special "UNK" token in the traning data.
  **Prediction(Viterbi):**
    Any word that was not present in the training vocabulary (or was replaced by "UNK" during training) is also mapped to the "UNK" token. The Viterbi algorithm then uses the emission probabilities learned for "UNK" to tag these unseen words.
    







**Smoothing:**
**Laplace:** is a technique used in probability estimation to handle the problem of zero probabilities.

Laplace smoothing addresses this by adding a small constant (typically 1, hence 'add-one') to each observed count, and also adding k (where k is the number of possible outcomes) to the denominator. This ensures that every possible event, even those not observed in the training data, receives a non-zero, albeit small, probability


In [23]:
class HMMTagger:
    """Vanilla HMM without UNK"""
    def __init__(self):
        self.transition_counts = defaultdict(lambda: defaultdict(int))
        self.emission_counts = defaultdict(lambda: defaultdict(int))
        self.state_counts = defaultdict(int)
        self.states = set()
        self.observations = set()

    def train(self, sequences):
        for state_seq, obs_seq in sequences:
            for i in range(len(state_seq)):
                state = state_seq[i]
                obs = obs_seq[i]
                self.emission_counts[state][obs] += 1
                self.state_counts[state] += 1
                self.states.add(state)
                self.observations.add(obs)
                if i < len(state_seq) - 1:
                    next_state = state_seq[i+1]
                    self.transition_counts[state][next_state] += 1
        self.states = list(self.states)
        self.observations = list(self.observations)

    def get_transition_probs(self, laplace=False):
        transition_probs = {}
        n_states = len(self.states)
        for s in self.states:
            total = sum(self.transition_counts[s].values())
            transition_probs[s] = {}
            for ns in self.states:
                count = self.transition_counts[s][ns]
                if laplace:
                    prob = (count + 1) / (total + n_states)
                else:
                    prob = count / total if total > 0 else 0
                transition_probs[s][ns] = prob
        return transition_probs

    def get_emission_probs(self, laplace=False):
        emission_probs = {}
        n_obs = len(self.observations)
        for s in self.states:
            total = self.state_counts[s]
            emission_probs[s] = {}
            for obs in self.observations:
                count = self.emission_counts[s][obs]
                if laplace:
                    prob = (count + 1) / (total + n_obs)
                else:
                    prob = count / total if total > 0 else 0
                emission_probs[s][obs] = prob
        return emission_probs

    def viterbi(self, obs_seq, transition_probs, emission_probs, initial_probs=None):
        """
        obs_seq: The sequence of observed words for which we want to find the most likely tag sequence.
        transition_probs: A dictionary containing the pre-calculated transition probabilities between states.
        emission_probs: A dictionary containing the pre-calculated emission probabilities of observations from states.
        initial_probs: An optional dictionary of initial probabilities for each state. If not provided,
                it defaults to a uniform distribution (1/n_states) across all states.
        """
        n_states = len(self.states)
        T = len(obs_seq)
        dp = np.zeros((n_states, T))
        backpointer = np.zeros((n_states, T), dtype=int)
        if initial_probs is None:
            initial_probs = {s: 1/n_states for s in self.states}
        # initialization
        for s_idx, s in enumerate(self.states):
            dp[s_idx, 0] = initial_probs[s] * emission_probs[s].get(obs_seq[0], 0)
        # recursion
        for t in range(1, T):
            for s_idx, s in enumerate(self.states):
                probs = []
                for ps_idx, ps in enumerate(self.states):
                    prob = dp[ps_idx, t-1] * transition_probs[ps][s] * emission_probs[s].get(obs_seq[t], 0)
                    probs.append(prob)
                backpointer[s_idx, t] = np.argmax(probs)
                dp[s_idx, t] = np.max(probs)
        # termination
        best_path = []
        last_state = np.argmax(dp[:, -1])
        best_path.append(self.states[last_state])
        for t in range(T-1, 0, -1):
            last_state = backpointer[last_state, t]
            best_path.insert(0, self.states[last_state])
        return best_path

In [24]:
class HMMTaggerUNK(HMMTagger):
    """HMM with UNK"""
    def __init__(self, unk_threshold=1):
        super().__init__()
        self.word_counts = defaultdict(int)
        self.unk_threshold = unk_threshold

    def train(self, sequences):
        # count words
        for _, obs_seq in sequences:
            for word in obs_seq:
                self.word_counts[word] += 1
        # replace rare words with UNK
        for state_seq, obs_seq in sequences:
            for i in range(len(state_seq)):
                state = state_seq[i]
                obs = obs_seq[i]
                if self.word_counts[obs] <= self.unk_threshold:
                    obs = "UNK"
                self.emission_counts[state][obs] += 1
                self.state_counts[state] += 1
                self.states.add(state)
                self.observations.add(obs)
                if i < len(state_seq) - 1:
                    next_state = state_seq[i+1]
                    self.transition_counts[state][next_state] += 1
        self.states = list(self.states)
        self.observations = list(self.observations)

    def viterbi(self, obs_seq, transition_probs, emission_probs, initial_probs=None):
        # map unseen words to UNK
        obs_seq = [w if w in self.observations else "UNK" for w in obs_seq]
        return super().viterbi(obs_seq, transition_probs, emission_probs, initial_probs)

In [25]:
#loading the treebank dataset
tagged_sents = treebank.tagged_sents(tagset='universal')
sequences = [([tag for (_, tag) in sent], [word.lower() for (word, _) in sent]) for sent in tagged_sents]
train_data, test_data = train_test_split(sequences, test_size=0.1, random_state=42)

In [26]:
#eveluting the HMM with transition, emission probabilities
def evaluate(hmm, laplace, test_data):
    tp = hmm.get_transition_probs(laplace=laplace)
    ep = hmm.get_emission_probs(laplace=laplace)
    print(tp, ep)
    correct, total = 0, 0
    for states_seq, obs_seq in test_data:
        pred_states = hmm.viterbi(obs_seq, tp, ep)
        for pred, gold in zip(pred_states, states_seq):
            if pred == gold:
                correct += 1
            total += 1
    return correct / total

In [27]:
# Train vanilla HMM
hmm_vanilla = HMMTagger()
hmm_vanilla.train(train_data)

In [28]:
# Train HMM with UNK
hmm_unk = HMMTaggerUNK(unk_threshold=1)
hmm_unk.train(train_data)

In [29]:
hmm_smoothing_unk = HMMTaggerUNK(unk_threshold=1)
hmm_smoothing_unk.train(train_data)

Below we are evaluating using three scenarios.
1. Vanilla HMM without UNK
2. HMM with UNK
3. HMM with UNK and adding Laplace smoothing.

In [30]:
acc_vanilla = evaluate(hmm_vanilla, laplace=False, test_data=test_data)
acc_unk = evaluate(hmm_unk, laplace=False, test_data=test_data)
acc_smoothing_unk = evaluate(hmm_smoothing_unk, laplace=True, test_data=test_data)

print(f"1. Vanilla HMM (no smoothing, no UNK): {acc_vanilla:.4f}")
print(f"2. HMM (no smoothing, with UNK):       {acc_unk:.4f}")
print(f"3. HMM (smoothing + UNK):              {acc_smoothing_unk:.4f}")


{'CONJ': {'CONJ': 0.0004962779156327543, 'NOUN': 0.35086848635235734, 'X': 0.00794044665012407, 'DET': 0.11861042183622829, 'ADP': 0.05508684863523573, 'ADJ': 0.11861042183622829, 'ADV': 0.05607940446650124, 'PRON': 0.05806451612903226, '.': 0.03573200992555831, 'NUM': 0.04168734491315137, 'PRT': 0.004466501240694789, 'VERB': 0.1523573200992556}, 'NOUN': {'CONJ': 0.04198886483142592, 'NOUN': 0.2637256418187442, 'X': 0.02911382616764615, 'DET': 0.013145685122177545, 'ADP': 0.17526291370244354, 'ADJ': 0.012179090627899784, 'ADV': 0.01666408908134859, 'PRON': 0.0046396535725332505, '.': 0.24122332199195792, 'NUM': 0.009665944942777605, 'PRT': 0.04442468295700588, 'VERB': 0.1479662851840396}, 'X': {'CONJ': 0.01076897189971395, 'NOUN': 0.060911997307757024, 'X': 0.07521453811206462, 'DET': 0.05552751135790005, 'ADP': 0.14319367322900892, 'ADJ': 0.015648662291771833, 'ADV': 0.02624936900555275, 'PRON': 0.05569577654383308, '.': 0.1640585562847047, 'NUM': 0.0028605081608615176, 'PRT': 0.18559

In [31]:
# Example sentence
#He attended IIID Generative AI Course
sentenses = ['Your Python code must be well-commented and clearly structured.', \
             'He attended IIID Generative AI Course.', \
             'Thoroughly explain the unknown word handling techniques implemented']
test_sentence = "He attended IIID Generative AI Course.".split(' ')

for each in sentenses:
  test_sentence = each.split(' ')
  print("Sentence: ", each)
  # Scenario 1: Vanilla HMM (no smoothing, no UNK)
  tp1 = hmm_vanilla.get_transition_probs(laplace=False)
  ep1 = hmm_vanilla.get_emission_probs(laplace=False)
  pred1 = hmm_vanilla.viterbi(test_sentence, tp1, ep1)
  print("Scenario 1 prediction:", pred1)

  # Scenario 2: HMM (no smoothing, with UNK)
  tp2 = hmm_unk.get_transition_probs(laplace=False)
  ep2 = hmm_unk.get_emission_probs(laplace=False)
  pred2 = hmm_unk.viterbi(test_sentence, tp2, ep2)
  print("Scenario 2 prediction:", pred2)

  # Scenario 3: HMM (smoothing + UNK)
  tp3 = hmm_smoothing_unk.get_transition_probs(laplace=True)
  ep3 = hmm_smoothing_unk.get_emission_probs(laplace=True)
  pred3 = hmm_smoothing_unk.viterbi(test_sentence, tp3, ep3)
  print("Scenario 3 prediction:", pred3)

Sentence:  Your Python code must be well-commented and clearly structured.
Scenario 1 prediction: ['CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ']
Scenario 2 prediction: ['ADJ', 'NOUN', 'NOUN', 'VERB', 'VERB', 'NOUN', 'CONJ', 'ADV', 'VERB']
Scenario 3 prediction: ['ADJ', 'NOUN', 'NOUN', 'VERB', 'VERB', 'NOUN', 'CONJ', 'ADV', 'VERB']
Sentence:  He attended IIID Generative AI Course.
Scenario 1 prediction: ['CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ']
Scenario 2 prediction: ['NUM', 'NUM', 'NUM', 'NUM', 'NUM', 'NOUN']
Scenario 3 prediction: ['ADJ', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN']
Sentence:  Thoroughly explain the unknown word handling techniques implemented
Scenario 1 prediction: ['CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ']
Scenario 2 prediction: ['VERB', 'VERB', 'DET', 'ADJ', 'NOUN', 'NOUN', 'NOUN', 'NOUN']
Scenario 3 prediction: ['NOUN', 'VERB', 'DET', 'NOUN', 'NOUN', 'NOUN', 'NOUN', 'NOUN']


**Quantitative Results (Accuracy):**
Here are the accuracy scores for the three scenarios on the test data:

1. Vanilla HMM (no smoothing, no UNK): 0.3003
2. HMM (no smoothing, with UNK): 0.9339
3. HMM (smoothing + UNK): 0.9073


**Impact of UNK Handling:** The most significant finding is the massive leap in accuracy from the Vanilla HMM (30%) to the HMM with UNK (93%). This clearly demonstrates that an explicit strategy for handling unknown words is absolutely critical for the practical application of POS tagging models. Without it, the model cannot assign probabilities to unseen words, leading to a breakdown in the Viterbi algorithm and extremely poor performance.

**Effect of Laplace Smoothing:** Surprisingly, adding Laplace smoothing to the HMM with UNK actually resulted in a slight decrease in accuracy (from ~93.39% to ~90.73%).

Qualitative Results (Example Sentence Predictions):
Look at the analysis of first sentence:

Sentence 1: "Your Python code must be well-commented and clearly structured."

Scenario 1 (Vanilla HMM): ['CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ', 'CONJ']
Scenario 2 (HMM with UNK): ['ADJ', 'NOUN', 'NOUN', 'VERB', 'VERB', 'NOUN', 'CONJ', 'ADV', 'VERB']
Scenario 3 (HMM with Smoothing + UNK): ['ADJ', 'NOUN', 'NOUN', 'VERB', 'VERB', 'NOUN', 'CONJ', 'ADV', 'VERB']
Discussion for Sentence 1:

Vanilla HMM: Predicts 'CONJ' for everything. This is a clear manifestation of the zero-probability problem. Many words like 'Python', 'well-commented', 'clearly', 'structured' are likely unseen, causing the model to default to a frequent tag or fail to make sensible predictions. The 30% accuracy is clearly reflected here.
HMM with UNK (both scenarios): Both models with UNK handling provide much more reasonable tag sequences.

**Final Observations**:
**Unknown Word Handling**: The most critical observation is the profound impact of explicitly handling unknown words. The accuracy jump from a mere 30% for the Vanilla HMM to over 93% with the UNK token clearly demonstrates that without a robust strategy for unseen words, an HMM-based POS tagger is practically unusable in real-world scenarios. The UNK token provides a crucial fallback, allowing the model to make educated guesses where it would otherwise fail entirely.


**Future Improvements**:

1.   **Advanced Smoothing Techniques**: Explore other smoothing methods beyond Laplace, such as Good-Turing smoothing or Kneser-Ney smoothing. These are often more sophisticated and can provide better probability estimates for unseen events without over-smoothing observed events.
2.   **Parameter Tuning**: Systematically tune the unk_threshold parameter (currently fixed at 1) using a validation set to find the optimal value that maximizes performance. This might reveal scenarios where a different threshold benefits the model.

